# Support Vector Machines
**Course:** Foundations of Machine Learning  
**Instructor:** Sayan CHAKI, LIRIS (UMR 5205 CNRS), École Centrale de Lyon, Université Lumière Lyon 2, INSA Lyon

**Lab 3.** Maximum margin, soft margin and C, hinge-loss SVM from scratch (Pegasos), dual coefficients, the kernel trick, RBF hyperparameters, multiclass SVM and SVR.

> Run in Google Colab: *Runtime → Run all*. All datasets ship with scikit-learn, so no download is needed.


## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
np.random.seed(0)
plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True

from sklearn.svm import SVC, LinearSVC, SVR
from sklearn.datasets import make_blobs, make_moons, make_circles

def plot_svm(clf, X, y, ax=None, title="", sv=True):
    ax = ax or plt.gca()
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min()-0.5, X[:, 0].max()+0.5, 300),
                         np.linspace(X[:, 1].min()-0.5, X[:, 1].max()+0.5, 300))
    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=20, cmap="coolwarm", alpha=0.4)
    ax.contour(xx, yy, Z, levels=[-1, 0, 1], colors="k", linestyles=["--", "-", "--"])
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=15, edgecolor="k", lw=0.3)
    if sv and hasattr(clf, "support_vectors_"):
        s = clf.support_vectors_
        ax.scatter(s[:, 0], s[:, 1], s=90, facecolors="none", edgecolors="k", lw=1)
    ax.set_title(title)

## 1. Maximum margin on separable data
Hard-margin SVM $\approx$ `SVC(kernel="linear", C=large)`. Dashed lines are the margins $\mathbf w^\top\mathbf x+b=\pm1$; circled points are support vectors.

In [ ]:
X, y = make_blobs(n_samples=60, centers=2, cluster_std=0.8, random_state=6)
clf = SVC(kernel="linear", C=1e5).fit(X, y)
w, b = clf.coef_[0], clf.intercept_[0]
plot_svm(clf, X, y, title=f"margin width = {2/np.linalg.norm(w):.3f}, #SV = {len(clf.support_)}")
plt.show()

### 1.1 Check the KKT structure
For a linear SVM, $\mathbf w=\sum_i \alpha_i y_i \mathbf x_i$. scikit-learn stores $\alpha_i y_i$ in `dual_coef_`.

In [ ]:
w_from_dual = clf.dual_coef_[0] @ clf.support_vectors_
print("w (primal):", w)
print("w (dual)  :", w_from_dual)
print("sum alpha_i y_i =", clf.dual_coef_.sum().round(6))
ys = np.where(y == 1, 1, -1)
margins = ys[clf.support_] * clf.decision_function(X[clf.support_])
print("y_i f(x_i) for support vectors:", margins.round(3))

## 2. Soft margin: the role of $C$

In [ ]:
X, y = make_blobs(n_samples=120, centers=2, cluster_std=1.8, random_state=2)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, Cval in zip(axes, [0.01, 1, 100]):
    m = SVC(kernel="linear", C=Cval).fit(X, y)
    plot_svm(m, X, y, ax=ax, title=f"C={Cval}  #SV={len(m.support_)}  acc={m.score(X, y):.2f}")
plt.show()

## 3. Linear SVM from scratch: sub-gradient descent on the hinge loss
$J(\mathbf w,b) = \frac\lambda2\|\mathbf w\|^2 + \frac1n\sum_i\max(0, 1-y_i(\mathbf w^\top\mathbf x_i+b))$  

Sub-gradient: $\lambda\mathbf w - \frac1n\sum_{i:\,y_if(\mathbf x_i)<1} y_i\mathbf x_i$ (and $-\frac1n\sum y_i$ for $b$). The hinge loss is not differentiable, so we use a decreasing step $\eta_t=\eta_0/\sqrt t$ and **iterate averaging**.

The reference is `SVC(kernel="linear", C=1/(lambda n))`, which solves exactly the same problem (the bias is not penalised).

In [ ]:
def hinge_svm(X, y, lam=0.01, iters=3000, lr0=1.0):
    y = np.where(y > 0, 1, -1)
    n, d = X.shape
    w, b = np.zeros(d), 0.0
    w_avg, b_avg, hist = np.zeros(d), 0.0, []
    for t in range(1, iters + 1):
        viol = y * (X @ w + b) < 1                       # points inside the margin or misclassified
        gw = lam * w - (y[viol, None] * X[viol]).sum(0) / n
        gb = -y[viol].sum() / n
        eta = lr0 / np.sqrt(t)
        w -= eta * gw; b -= eta * gb
        w_avg += (w - w_avg) / t; b_avg += (b - b_avg) / t   # running average
        hist.append(lam / 2 * w_avg @ w_avg + np.mean(np.maximum(0, 1 - y * (X @ w_avg + b_avg))))
    return w_avg, b_avg, hist

Xs = StandardScaler().fit_transform(X)
lam = 0.01
w_h, b_h, hist = hinge_svm(Xs, y, lam=lam)
plt.plot(hist); plt.xscale("log"); plt.xlabel("iteration"); plt.ylabel("primal objective"); plt.show()

ref = SVC(kernel="linear", C=1 / (lam * len(Xs))).fit(Xs, y)
print("scratch : w =", w_h.round(3), " b =", round(b_h, 3))
print("SVC     : w =", ref.coef_[0].round(3), " b =", ref.intercept_[0].round(3))
plot_svm(ref, Xs, y, title="Reference solution"); plt.show()

### 3.1 Comparing surrogate losses

In [ ]:
m = np.linspace(-2.5, 3, 300)
plt.plot(m, (m <= 0).astype(float), "k", label="0-1")
plt.plot(m, np.maximum(0, 1 - m), label="hinge (SVM)")
plt.plot(m, np.log2(1 + np.exp(-m)), label="logistic")
plt.plot(m, np.maximum(0, 1 - m) ** 2, label="squared hinge")
plt.ylim(0, 4); plt.xlabel("margin  y f(x)"); plt.legend(); plt.show()

## 4. The kernel trick
Data that are not linearly separable can become separable in a feature space. We first do it **explicitly**, then with a kernel.

In [ ]:
Xc, yc = make_circles(n_samples=300, factor=0.4, noise=0.08, random_state=0)
phi = np.c_[Xc, (Xc ** 2).sum(1)]           # explicit map (x1, x2, x1^2 + x2^2)

fig = plt.figure(figsize=(12, 4.5))
ax1 = fig.add_subplot(1, 2, 1); ax1.scatter(Xc[:, 0], Xc[:, 1], c=yc, cmap="coolwarm", s=12); ax1.set_title("input space")
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
ax2.scatter(phi[:, 0], phi[:, 1], phi[:, 2], c=yc, cmap="coolwarm", s=12); ax2.set_title("feature space")
plt.show()

print("linear SVM on explicit features, acc =", SVC(kernel="linear").fit(phi, yc).score(phi, yc))

### 4.1 Verifying a kernel identity
For $k(\mathbf x,\mathbf z)=(\mathbf x^\top\mathbf z)^2$ in 2D, $\phi(\mathbf x)=(x_1^2,\sqrt2x_1x_2,x_2^2)$.

In [ ]:
def phi_poly2(x):
    return np.array([x[0]**2, np.sqrt(2) * x[0] * x[1], x[1]**2])
a, c = np.random.randn(2), np.random.randn(2)
print("kernel      :", (a @ c) ** 2)
print("inner prod. :", phi_poly2(a) @ phi_poly2(c))

# Gram matrices are PSD
from sklearn.metrics.pairwise import rbf_kernel
K = rbf_kernel(np.random.randn(50, 3), gamma=0.5)
print("min eigenvalue of RBF Gram matrix:", np.linalg.eigvalsh(K).min())

### 4.2 Comparing kernels

In [ ]:
Xm, ym = make_moons(n_samples=300, noise=0.2, random_state=0)
Xm = StandardScaler().fit_transform(Xm)
kernels = [("linear", dict(kernel="linear")),
           ("poly, d=3", dict(kernel="poly", degree=3, coef0=1)),
           ("RBF", dict(kernel="rbf", gamma=1.0)),
           ("sigmoid", dict(kernel="sigmoid", gamma=0.5, coef0=0))]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, kw) in zip(axes, kernels):
    m = SVC(C=1, **kw).fit(Xm, ym)
    cv = cross_val_score(SVC(C=1, **kw), Xm, ym, cv=5).mean()
    plot_svm(m, Xm, ym, ax=ax, title=f"{name}: CV acc={cv:.2f}", sv=False)
plt.show()

### 4.3 RBF: the $(C,\gamma)$ grid
$\gamma$ sets the width of each Gaussian bump, $C$ the penalty on violations.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 11))
for i, g in enumerate([0.1, 1, 10]):
    for j, Cval in enumerate([0.1, 1, 100]):
        m = SVC(kernel="rbf", gamma=g, C=Cval).fit(Xm, ym)
        plot_svm(m, Xm, ym, ax=axes[i, j], title=f"gamma={g}, C={Cval}, #SV={len(m.support_)}", sv=False)
plt.tight_layout(); plt.show()

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(Xm, ym, test_size=0.3, random_state=0)
Cs, gammas = np.logspace(-2, 3, 6), np.logspace(-3, 2, 6)
grid = GridSearchCV(make_pipeline(StandardScaler(), SVC(kernel="rbf")),
                    {"svc__C": Cs, "svc__gamma": gammas}, cv=5).fit(Xtr, ytr)
print("best:", grid.best_params_, " test acc:", grid.score(Xte, yte))

scores = grid.cv_results_["mean_test_score"].reshape(len(Cs), len(gammas))
plt.imshow(scores, cmap="viridis", origin="lower")
plt.xticks(range(len(gammas)), [f"{g:g}" for g in gammas]); plt.yticks(range(len(Cs)), [f"{c:g}" for c in Cs])
plt.xlabel("gamma"); plt.ylabel("C"); plt.colorbar(label="CV accuracy"); plt.grid(False)
plt.title("Validation accuracy heat-map"); plt.show()

## 5. Real data: breast cancer, scaling matters

In [ ]:
from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(bc.data, bc.target, stratify=bc.target, test_size=0.25, random_state=0)
print("RBF SVM without scaling:", SVC().fit(Xtr, ytr).score(Xte, yte))
print("RBF SVM with scaling   :", make_pipeline(StandardScaler(), SVC()).fit(Xtr, ytr).score(Xte, yte))

## 6. Multiclass SVM on digits

In [ ]:
from sklearn.datasets import load_digits
from sklearn.metrics import ConfusionMatrixDisplay
import time
dg = load_digits()
Xtr, Xte, ytr, yte = train_test_split(dg.data, dg.target, stratify=dg.target, test_size=0.25, random_state=0)
for name, model in [("LinearSVC (OvR)", make_pipeline(StandardScaler(), LinearSVC(max_iter=20000))),
                    ("SVC RBF (OvO)", make_pipeline(StandardScaler(), SVC(gamma="scale", C=10)))]:
    t = time.time(); model.fit(Xtr, ytr)
    print(f"{name:16s} acc={model.score(Xte, yte):.4f}  time={time.time()-t:.2f}s")
ConfusionMatrixDisplay.from_estimator(model, Xte, yte, cmap="Blues"); plt.grid(False); plt.show()

## 7. Support Vector Regression
$\varepsilon$-insensitive loss: errors smaller than $\varepsilon$ are ignored, giving a tube around the prediction.

In [ ]:
rng = np.random.default_rng(0)
xr = np.sort(rng.uniform(0, 5, 80))[:, None]
yr = np.sin(xr).ravel() + rng.normal(0, 0.15, 80)
grid_x = np.linspace(0, 5, 300)[:, None]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, eps in zip(axes, [0.05, 0.2, 0.5]):
    svr = SVR(kernel="rbf", C=10, epsilon=eps).fit(xr, yr)
    pr = svr.predict(grid_x)
    ax.scatter(xr, yr, s=12, c="gray")
    ax.scatter(xr[svr.support_], yr[svr.support_], s=40, facecolors="none", edgecolors="r")
    ax.plot(grid_x, pr, "b"); ax.fill_between(grid_x.ravel(), pr - eps, pr + eps, alpha=0.2)
    ax.set_title(f"epsilon={eps}, #SV={len(svr.support_)}")
plt.show()

## 8. Exercises
1. Solve the **dual** QP for a small dataset with `scipy.optimize.minimize` (SLSQP) or `cvxpy`, recover $\mathbf w$ and $b$, and compare with `SVC(kernel="linear")`.
2. Implement a kernelised prediction $f(\mathbf x)=\sum_i\alpha_iy_ik(\mathbf x_i,\mathbf x)+b$ using `dual_coef_`, `support_vectors_` and `intercept_` from a fitted RBF `SVC`, and check it matches `decision_function`.
3. Use `sklearn.kernel_approximation.Nystroem` + `LinearSVC` on digits and compare time/accuracy with `SVC`.
4. Create an imbalanced dataset (`weights=[0.95, 0.05]`) and study `class_weight="balanced"`.

In [ ]:
# Exercise 2 (starter)
m = SVC(kernel="rbf", gamma=1.0, C=1).fit(Xm, ym)
x_new = Xm[:5]
K = rbf_kernel(x_new, m.support_vectors_, gamma=1.0)
f_manual = K @ m.dual_coef_[0] + m.intercept_[0]
print(np.allclose(f_manual, m.decision_function(x_new)))